# Zero-Shot Phase-Space CASAL Projected Velocity Tests

This notebook tests the decisive phase-space question: can a fixed-template CASAL step reduce both coordinate residual `D_f` and tangent-velocity residual `D_v` for KLDM states?

In [1]:
import dataclasses
import math
import os
import random
import sys
import time
import traceback
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Optional

import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'configs').exists():
    ROOT = ROOT.parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from kldmPlus.algorithm10_casal_chart import Algorithm10Config, _decode_lattice_matrix, _encode_lattice_matrix, _k_to_l, _l_to_k
from kldmPlus.algorithm13_fixed_template_velocity_casal import (
    FixedTemplateVelocityConfig,
    apply_full_space_force,
    apply_reduced_space_force,
    build_cartesian_block_metric,
    build_fractional_metric,
    center_velocity,
    compute_template_jacobian,
    finite_difference_jacobian_error,
    graph_velocity_norm,
    kinetic_position_update,
    lift_reduced_chart_velocity,
    mean_norm,
    project_to_fixed_template,
    project_to_fixed_template_local,
    reduced_chart_velocity,
    tangent_project_velocity,
    tangent_projector,
    wrap_residual,
)
from kldmPlus.algorithm14_kldm_casal_velocity_impulse import (
    CASAL_THEOREM_APPLIES,
    CASAL_THEOREM_REASON,
    center_velocity as center_velocity_impulse,
    kinetic_position_update_signed,
    tangent_project_mean_free,
    wrapdiff,
)
from kldmPlus.algorithm15_zero_shot_phase_space_casal import (
    PhaseSpaceCasalConfig,
    phase_space_casal_step,
    project_state_to_phase_space_constraint,
)
from kldmPlus.run_sampling_compare import SamplingCompareRunner, set_seed
from kldmPlus.sample_evaluation import evaluate_csp_reconstruction
from kldmPlus.symmetry.frame_bridge import map_standardized_structure_to_vanilla_frame, standardize_structure
from kldmPlus.symmetry import (
    materialize_pcs_state,
    sample_random_free_vars,
    select_requested_template_state,
    select_requested_template_states,
    validate_requested_space_group,
)
from kldmPlus.symmetry.pcs_projection import (
    _build_vanilla_structure,
    _collapse_centering_equivalent_structure,
    _periodic_pairwise_distances,
    refresh_pcs_state_anchor,
    vanilla_structure_to_model_tensors,
)
from kldmPlus.utils.time import iter_sampling_times

CONFIG_PATH = ROOT / 'configs' / 'kldm_plus' / 'mp_20' / 'mp20_sampling_compare_em_vs_algorithm10_casal_chart.yaml'
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)
COMPARE_CFG = CONFIG['sampling_compare']

PROFILE = os.environ.get('PHASE_SPACE_CASAL_PROFILE', os.environ.get('FIXED_TEMPLATE_VELOCITY_PROFILE', 'laptop')).strip().lower()

def profile_default(name: str, laptop: str, deep: str | None = None) -> str:
    if name in os.environ:
        return os.environ[name]
    if PROFILE in {'full', 'deep'} and deep is not None:
        return deep
    return laptop


def parse_int_list(text: str) -> list[int]:
    return [int(v.strip()) for v in str(text).split(',') if v.strip()]


def parse_float_list(text: str) -> list[float]:
    return [float(v.strip()) for v in str(text).split(',') if v.strip()]

SAMPLE_SEED = int(os.environ.get('PHASE_SPACE_CASAL_SEED', os.environ.get('FIXED_TEMPLATE_VELOCITY_SEED', profile_default('PHASE_SPACE_CASAL_SEED', '7', '7'))))
GRAPH_IDS = parse_int_list(os.environ.get('PHASE_SPACE_CASAL_GRAPH_IDS', os.environ.get('FIXED_TEMPLATE_VELOCITY_GRAPH_IDS', profile_default('PHASE_SPACE_CASAL_GRAPH_IDS', '1,2,3', '1,2,3'))))
CAPTURE_N_STEPS = int(os.environ.get('PHASE_SPACE_CASAL_CAPTURE_N_STEPS', os.environ.get('FIXED_TEMPLATE_VELOCITY_CAPTURE_N_STEPS', profile_default('PHASE_SPACE_CASAL_CAPTURE_N_STEPS', '80', '160'))))
CAPTURE_STEP = int(os.environ.get('PHASE_SPACE_CASAL_CAPTURE_STEP', os.environ.get('FIXED_TEMPLATE_VELOCITY_CAPTURE_STEP', profile_default('PHASE_SPACE_CASAL_CAPTURE_STEP', '72', '144'))))
MINI_CHAIN_STEPS = int(os.environ.get('PHASE_SPACE_CASAL_MINI_CHAIN_STEPS', os.environ.get('FIXED_TEMPLATE_VELOCITY_MINI_CHAIN_STEPS', profile_default('PHASE_SPACE_CASAL_MINI_CHAIN_STEPS', '12', '20'))))
FULL_CHAIN_STEPS = int(os.environ.get('PHASE_SPACE_CASAL_FULL_CHAIN_STEPS', os.environ.get('FIXED_TEMPLATE_VELOCITY_FULL_CHAIN_STEPS', profile_default('PHASE_SPACE_CASAL_FULL_CHAIN_STEPS', '40', '80'))))
LATE_START_STEP = int(os.environ.get('PHASE_SPACE_CASAL_LATE_START_STEP', os.environ.get('FIXED_TEMPLATE_VELOCITY_LATE_START_STEP', profile_default('PHASE_SPACE_CASAL_LATE_START_STEP', '32', '64'))))
FIXTURE_MAX_TEMPLATES = int(os.environ.get('PHASE_SPACE_CASAL_MAX_TEMPLATES', os.environ.get('FIXED_TEMPLATE_VELOCITY_MAX_TEMPLATES', profile_default('PHASE_SPACE_CASAL_MAX_TEMPLATES', '12', '20'))))
FIXTURE_TEMPLATE_EVAL_LIMIT = int(os.environ.get('PHASE_SPACE_CASAL_TEMPLATE_EVAL_LIMIT', os.environ.get('FIXED_TEMPLATE_VELOCITY_TEMPLATE_EVAL_LIMIT', profile_default('PHASE_SPACE_CASAL_TEMPLATE_EVAL_LIMIT', '4', '6'))))
FIXTURE_OPT_STEPS = int(os.environ.get('PHASE_SPACE_CASAL_OPT_STEPS', os.environ.get('FIXED_TEMPLATE_VELOCITY_OPT_STEPS', profile_default('PHASE_SPACE_CASAL_OPT_STEPS', '80', '120'))))
FIXTURE_LR = float(os.environ.get('PHASE_SPACE_CASAL_LR', os.environ.get('FIXED_TEMPLATE_VELOCITY_LR', profile_default('PHASE_SPACE_CASAL_LR', '5e-2', '5e-2'))))
TEMPLATE_TOP_K = int(os.environ.get('PHASE_SPACE_CASAL_TEMPLATE_TOP_K', os.environ.get('FIXED_TEMPLATE_VELOCITY_TEMPLATE_TOP_K', profile_default('PHASE_SPACE_CASAL_TEMPLATE_TOP_K', '3', '5'))))
ORACLE_W_MODE = str(os.environ.get('PHASE_SPACE_CASAL_ORACLE_W_MODE', os.environ.get('FIXED_TEMPLATE_VELOCITY_ORACLE_W_MODE', profile_default('PHASE_SPACE_CASAL_ORACLE_W_MODE', '1', '1')))).strip().lower() not in {'0', 'false', 'no'}
EPS_PASS = 1.0e-6
FD_EPS_VALUES = parse_float_list(profile_default('PHASE_SPACE_CASAL_FD_EPS', '1e-2,1e-3,1e-4', '1e-2,1e-3,1e-4'))
RHO_F_SWEEP = parse_float_list(profile_default('PHASE_SPACE_CASAL_RHO_F', '0.1,1.0,3.0,10.0', '0.1,1.0,3.0,10.0'))
KAPPA_SWEEP = parse_float_list(profile_default('PHASE_SPACE_CASAL_KAPPA', '0.01,0.03,0.1,0.3', '0.01,0.03,0.1,0.3'))
DUAL_MODES = [part.strip() for part in profile_default('PHASE_SPACE_CASAL_DUAL_MODES', 'no_dual,mu_f_only,mu_f_and_mu_v', 'no_dual,mu_f_only,mu_f_and_mu_v').split(',') if part.strip()]
print(f'profile={PROFILE} graphs={GRAPH_IDS} capture_step={CAPTURE_STEP}/{CAPTURE_N_STEPS} mini_chain_steps={MINI_CHAIN_STEPS} full_chain_steps={FULL_CHAIN_STEPS}')


MODELS_PROJECT_ROOT: /workspace/.venv/lib/python3.11/site-packages/mattergen
profile=laptop graphs=[1, 2, 3] capture_step=72/80 mini_chain_steps=12 full_chain_steps=40


In [2]:
set_seed(SAMPLE_SEED)
runner = SamplingCompareRunner(config_path=CONFIG_PATH)
runner.model.eval()
template_prior = runner._ensure_template_prior()
batch = next(iter(runner.loader)).to(runner.device)
ptr = batch.ptr.tolist()

sampler_cfgs = {cfg['name']: dict(cfg) for cfg in COMPARE_CFG['samplers']}
facit_cfg = sampler_cfgs.get('FacitKLDM_PC', COMPARE_CFG['samplers'][0])
algo10_cfg_map = dict(sampler_cfgs['Algorithm10_CASAL_chart'].get('algorithm10', {}))
BASE_ALGO10 = Algorithm10Config.from_mapping(algo10_cfg_map)


def replace_supported_dataclass(obj, **updates):
    supported = {field.name for field in dataclasses.fields(obj)}
    return dataclasses.replace(obj, **{key: value for key, value in updates.items() if key in supported})


BASE_GUIDE_ALGO10 = replace_supported_dataclass(
    BASE_ALGO10,
    debug=False,
    oracle_wyckoff_debug=True,
    return_best_even_if_invalid=True,
    max_templates=FIXTURE_MAX_TEMPLATES,
    template_eval_limit=FIXTURE_TEMPLATE_EVAL_LIMIT,
    ssvd_max_steps=min(int(getattr(BASE_ALGO10, 'ssvd_max_steps', 8)), 6),
    ssvd_random_restarts=0,
)

FIXED_CFG = FixedTemplateVelocityConfig(projector_damping=1.0e-6)

@dataclass
class GraphCase:
    graph_id: int
    graph_idx0: int
    composition: dict[int, int]
    atomic_numbers: torch.Tensor
    gt_frac: torch.Tensor
    gt_lattice: torch.Tensor
    requested_sg: int


@dataclass
class KLDMState:
    f: torch.Tensor
    v: torch.Tensor
    l: torch.Tensor
    k: torch.Tensor
    h: torch.Tensor
    t: float
    dt: float
    graph_idx0: int
    full_state: Optional[dict[str, Any]] = None
    full_times: Optional[Any] = None


@dataclass
class FixedTemplateFixture:
    case: GraphCase
    state: Any
    theta: torch.Tensor
    tau: torch.Tensor
    z_frac: torch.Tensor
    z_k: torch.Tensor
    z_l: torch.Tensor
    assignment: torch.Tensor
    J: torch.Tensor
    template_labels: str
    template_multiplicities: str
    template_dofs: str
    template_total_dof: int
    requested_sg_match: bool
    composition_match: bool
    projection_objective: float
    projection_rank: int | None
    projection_condition: float | None
    config: FixedTemplateVelocityConfig
    chart_frac: torch.Tensor | None = None
    chart_atomic_numbers: torch.Tensor | None = None
    chart_l: torch.Tensor | None = None
    chart_k: torch.Tensor | None = None
    chart_num_atoms: int = 0
    raw_num_atoms: int = 0
    target_representation_name: str = ''
    anchor_representation_name: str = ''
    uses_oracle_chart: bool = False
    oracle_w_payload: dict[str, Any] | None = None


result_tables: dict[str, pd.DataFrame] = {}
_caches: dict[tuple[Any, ...], Any] = {}


def graph_slice(graph_idx0: int) -> tuple[int, int]:
    return int(ptr[graph_idx0]), int(ptr[graph_idx0 + 1])


def composition_counter(values) -> dict[int, int]:
    arr = [int(v) for v in torch.as_tensor(values, dtype=torch.long).reshape(-1).tolist()]
    return dict(sorted(Counter(arr).items()))


def graph_tensors(graph_idx0: int, *, source=None) -> dict[str, Any]:
    start, end = graph_slice(graph_idx0)
    if source is None:
        pos, l, h = batch.pos, batch.l, batch.atomic_numbers
    else:
        if len(source) == 4:
            pos, _v, l, h = source
        else:
            pos, l, h = source
    return {
        'pos': pos[start:end].detach().clone(),
        'l': l[graph_idx0].detach().clone().reshape(-1),
        'h': h[start:end].detach().clone().to(torch.long),
        'sg': int(torch.as_tensor(batch.space_group).reshape(-1)[graph_idx0].item()),
        'num_atoms': int(end - start),
    }


def load_test_graphs(graph_ids=GRAPH_IDS) -> list[GraphCase]:
    out = []
    num_graphs_in_batch = len(ptr) - 1
    for graph_id in graph_ids:
        graph_idx0 = int(graph_id) - 1
        if graph_idx0 < 0 or graph_idx0 >= num_graphs_in_batch:
            raise IndexError(f'graph_id={graph_id} is outside the loaded batch (num_graphs_in_batch={num_graphs_in_batch})')
        g = graph_tensors(graph_idx0)
        out.append(
            GraphCase(
                graph_id=int(graph_id),
                graph_idx0=graph_idx0,
                composition=composition_counter(g['h']),
                atomic_numbers=g['h'],
                gt_frac=g['pos'],
                gt_lattice=g['l'],
                requested_sg=g['sg'],
            )
        )
    return out


GRAPH_CASES = load_test_graphs()
backend_summary = {
    'state_backend': 'KLDM reverse sampler via SamplingCompareRunner/model._prepare_csp_sampling + reverse_exp_step',
    'projection_backend': 'fixed_template_ssvd_project via algorithm14_kldm_casal_velocity_impulse (faithful velocity impulse + CASCAL z/mu)',
    'oracle_w_source': 'oracle structure -> select_requested_template_states -> PCSTemplateState/template extractor -> Phi_W/J_W',
    'graph_import_backend': 'identical_to_test_kldm_casal_velocity_impulse',
}
print(f'loaded_graphs={[g.graph_id for g in GRAPH_CASES]} requested_sg={[g.requested_sg for g in GRAPH_CASES]}')
print('backend_summary=', backend_summary)

print(f'CASAL_THEOREM_APPLIES={CASAL_THEOREM_APPLIES} REASON={CASAL_THEOREM_REASON}')


mps:  False
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test


dataset_cache action=builder_build:done dataset=mp_20 split=test
dataset_cache action=load dataset=mp_20 split=train reason=fresh path=/workspace/data/mp_20/processed/train
dataset_cache action=from_cache_path:start dataset=mp_20 split=train
dataset_cache action=from_cache_path:done dataset=mp_20 split=train
dataset_cache action=builder_build:start dataset=mp_20 split=train
dataset_cache action=builder_build:done dataset=mp_20 split=train
template_prior_cache_hit path=/workspace/notebooks/.cache/kldmPlus/template_prior/0448db66e575a7bb.pkl records=43
loaded_graphs=[1, 2, 3] requested_sg=[227, 4, 194]
backend_summary= {'state_backend': 'KLDM reverse sampler via SamplingCompareRunner/model._prepare_csp_sampling + reverse_exp_step', 'projection_backend': 'fixed_template_ssvd_project via algorithm14_kldm_casal_velocity_impulse (faithful velocity impulse + CASCAL z/mu)', 'oracle_w_source': 'oracle structure -> select_requested_template_states -> PCSTemplateState/template extractor -> Phi_W/

In [3]:
def dataframe_result(name: str, rows: list[dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    if 'PASS' not in df.columns:
        df['PASS'] = False
    if 'status' not in df.columns:
        df['status'] = np.where(df['PASS'], 'PASS', 'FAIL')
    result_tables[name] = df
    return df


def error_row(test: str, graph: int | str, exc: Exception, failure_category: str, **extra) -> dict[str, Any]:
    tb = traceback.format_exc().strip().splitlines()
    return {
        'test': test,
        'graph': graph,
        'PASS': False,
        'status': 'ERROR',
        'error_type': type(exc).__name__,
        'error_message': str(exc),
        'traceback_tail': tb[-1] if tb else '',
        'failure_category': failure_category,
        **extra,
    }


def print_group_pass(df: pd.DataFrame, group_cols) -> None:
    if df.empty:
        print('empty')
        return
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    cols = list(group_cols) + ['PASS', 'status']
    display(df[cols].groupby(group_cols, as_index=False).agg({'PASS': 'all', 'status': 'last'}))


def l_to_k(l: torch.Tensor, graph_case: GraphCase) -> torch.Tensor:
    return _l_to_k(
        l=l.reshape(-1),
        num_atoms=int(graph_case.gt_frac.shape[0]),
        lattice_transform=runner.lattice_transform,
    ).to(device=l.device, dtype=l.dtype)


def k_to_l(k: torch.Tensor, graph_case: GraphCase) -> torch.Tensor:
    return _k_to_l(
        k=k.reshape(-1),
        num_atoms=int(graph_case.gt_frac.shape[0]),
        lattice_transform=runner.lattice_transform,
    ).to(device=k.device, dtype=k.dtype)


def ensure_l_feature(l: torch.Tensor, num_atoms: int) -> torch.Tensor:
    flat = l.reshape(-1)
    if int(flat.numel()) == 9:
        encoded = _encode_lattice_matrix(
            cell_matrix=flat.reshape(3, 3),
            num_atoms=int(num_atoms),
            lattice_transform=runner.lattice_transform,
        )
        return encoded.to(device=l.device, dtype=l.dtype).reshape(-1)
    return flat


def cell_from_l(l: torch.Tensor, num_atoms: int) -> torch.Tensor:
    flat = l.reshape(-1)
    if int(flat.numel()) == 9:
        return flat.reshape(3, 3).detach()
    return _decode_lattice_matrix(
        l=flat,
        num_atoms=int(num_atoms),
        lattice_transform=runner.lattice_transform,
    ).detach()


def volume_from_l(l: torch.Tensor, num_atoms: int) -> float:
    return float(torch.abs(torch.linalg.det(cell_from_l(l, num_atoms))).detach().item())


def min_pair_distance_from_arrays(frac: torch.Tensor, l: torch.Tensor, num_atoms: int) -> float:
    cell = cell_from_l(l, num_atoms).to(device=frac.device, dtype=frac.dtype)
    distances = _periodic_pairwise_distances(frac_coords=frac, cell_matrix=cell)
    return float(distances.min().detach().item()) if distances.numel() else float('inf')


def cart_distance_to_projection(frac: torch.Tensor, projection, num_atoms: int) -> float:
    cell = cell_from_l(projection.z_l, num_atoms).to(device=frac.device, dtype=frac.dtype)
    delta = wrap_residual(frac, projection.z_frac) @ cell
    return float(torch.linalg.norm(delta.reshape(-1)).detach().item())


def clone_full_state(full_state: dict[str, Any]) -> dict[str, Any]:
    cloned: dict[str, Any] = {}
    for key, value in full_state.items():
        cloned[key] = value.clone() if torch.is_tensor(value) else value
    return cloned


def _native_reverse_step_full_state(full_state: dict[str, Any], times) -> dict[str, Any]:
    with torch.no_grad():
        preds_curr = full_state['score_network'](
            t=times.now.graph,
            pos=full_state['f_t'],
            v=full_state['v_t'],
            h=full_state['a_t'],
            l=full_state['l_t'],
            node_index=full_state['node_index'],
            edge_node_index=full_state['edge_node_index'],
        )
        score_v = full_state['sampling_tdm'].reconstruct_full_reverse_velocity_score(
            t=times.now.nodes,
            v_t=full_state['v_t'],
            pred_v=preds_curr['v'],
            index=full_state['node_index'],
        )
        full_state['f_t'], full_state['v_t'] = full_state['sampling_tdm'].reverse_exp_step(
            f_t=full_state['f_t'],
            v_t=full_state['v_t'],
            score_v=score_v,
            index=full_state['node_index'],
            dt=times.dt,
        )
        full_state['l_t'] = runner.model._reverse_lattice_sampling_step(
            t=times.now.lattice,
            x_t=full_state['l_t'],
            pred=preds_curr['l'],
            dt=times.dt,
            num_atoms=batch.num_atoms,
        )
    return full_state


def graph_state_from_full(full_state: dict[str, Any], case: GraphCase, times=None) -> KLDMState:
    start, end = graph_slice(case.graph_idx0)
    f = full_state['f_t'][start:end].detach().clone()
    v = full_state['v_t'][start:end].detach().clone()
    l = full_state['l_t'][case.graph_idx0].detach().clone().reshape(-1)
    h = full_state['a_t'][start:end].detach().clone().to(torch.long)
    k = l_to_k(l, case)
    dt = float(times.dt) if times is not None else 1.0 / max(1, CAPTURE_N_STEPS)
    t = float(times.now.graph[case.graph_idx0].detach().reshape(-1)[0].item()) if times is not None else float('nan')
    return KLDMState(f=f, v=v, l=l, k=k, h=h, t=t, dt=dt, graph_idx0=case.graph_idx0, full_state=full_state, full_times=times)


def capture_batch_kldm_state(seed=SAMPLE_SEED, n_steps=CAPTURE_N_STEPS, capture_step=CAPTURE_STEP):
    key = ('capture', int(seed), int(n_steps), int(capture_step))
    if key in _caches:
        return _caches[key]
    set_seed(seed)
    state = runner.model._prepare_csp_sampling(
        batch=batch,
        n_steps=int(n_steps),
        t_start=float(facit_cfg.get('t_start', 1.0)),
        t_final=float(facit_cfg.get('t_final', 1e-3)),
    )
    last_times = None
    for step_idx, times in enumerate(iter_sampling_times(batch=state['batch'], grid=state['sampling_time_grid']), start=1):
        last_times = times
        state = _native_reverse_step_full_state(state, times)
        if step_idx >= int(capture_step):
            break
    _caches[key] = (state, last_times, int(capture_step))
    return _caches[key]

# Canonical state-lifting, evaluation, and score-compatibility helpers live in Cell 5.


In [4]:


def metric_tensor(metric_name: str, l: torch.Tensor, num_atoms: int) -> torch.Tensor | None:
    metric_name = str(metric_name).strip().lower()
    if metric_name == 'cartesian':
        cell = cell_from_l(l, num_atoms).to(device=l.device, dtype=l.dtype)
        return build_cartesian_block_metric(cell, num_atoms=num_atoms)
    return build_fractional_metric(num_atoms=num_atoms, dtype=l.dtype, device=l.device)


def template_labels(template) -> str:
    return ','.join(f"{site.atomic_number}@{site.label}" for site in template.site_templates)


def template_multiplicities(template) -> str:
    return ','.join(str(int(site.multiplicity)) for site in template.site_templates)


def template_dofs(template) -> str:
    return ','.join(str(int(site.dof)) for site in template.site_templates)


def l_to_k_num_atoms(l: torch.Tensor, num_atoms: int) -> torch.Tensor:
    return _l_to_k(
        l=ensure_l_feature(l, int(num_atoms)).reshape(-1),
        num_atoms=int(num_atoms),
        lattice_transform=runner.lattice_transform,
    ).to(device=l.device, dtype=l.dtype)


def k_to_l_num_atoms(k: torch.Tensor, num_atoms: int) -> torch.Tensor:
    return _k_to_l(
        k=k.reshape(-1),
        num_atoms=int(num_atoms),
        lattice_transform=runner.lattice_transform,
    ).to(device=k.device, dtype=k.dtype)


def _pcs_state_chart_target(state0) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, str, str]:
    chart_frac = state0.target_frac if state0.target_frac is not None else state0.anchor_frac
    chart_atomic_numbers = state0.target_atomic_numbers if state0.target_atomic_numbers is not None else state0.anchor_atomic_numbers
    chart_cell = state0.target_cell if state0.target_cell is not None else state0.anchor_cell
    chart_k = state0.target_k if state0.target_k is not None else state0.anchor_k
    if chart_frac is None or chart_atomic_numbers is None or chart_cell is None or chart_k is None:
        raise RuntimeError('PCS state is missing oracle chart tensors.')
    return (
        chart_frac.detach().clone(),
        chart_atomic_numbers.detach().clone().to(torch.long),
        chart_cell.detach().clone(),
        chart_k.detach().clone(),
        str(state0.target_representation_name or 'standardized'),
        str(state0.anchor_representation_name or 'standardized'),
    )


def _oracle_w_payload_from_state(case: GraphCase, state0, *, chart_num_atoms: int, chart_repr: str, anchor_repr: str) -> dict[str, Any]:
    return {
        'space_group': int(case.requested_sg),
        'template_labels': template_labels(state0.template),
        'template_multiplicities': template_multiplicities(state0.template),
        'template_dofs': template_dofs(state0.template),
        'template_total_dof': int(state0.template.total_free_dims),
        'template_num_atoms': int(state0.template.total_atoms),
        'chart_num_atoms': int(chart_num_atoms),
        'raw_num_atoms': int(case.gt_frac.shape[0]),
        'target_representation': chart_repr,
        'anchor_representation': anchor_repr,
        'species_assignment_mode': 'oracle_fixed_template',
        'origin_gauge_mode': 'pcs_anchor',
        'extraction_backend': 'select_requested_template_states -> PCSTemplateState',
        'jacobian_backend': 'compute_template_jacobian(materialize_template)',
        'projection_backend': 'fixed_template_ssvd_project',
        'state_backend': 'KLDM reverse sampler',
    }


def _chart_structure_to_vanilla_tensors(
    case: GraphCase,
    fixture: FixedTemplateFixture,
    *,
    frac_coords: torch.Tensor,
    atomic_numbers: torch.Tensor,
    l: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    cell = cell_from_l(l, int(frac_coords.shape[0])).to(device=frac_coords.device, dtype=frac_coords.dtype)
    structure = _build_vanilla_structure(
        frac_coords=frac_coords,
        atomic_numbers=atomic_numbers,
        cell_matrix=cell,
    )
    standardized_like = structure
    translations = fixture.state.target_centering_translations
    if translations is not None and int(atomic_numbers.shape[0]) != int(case.atomic_numbers.shape[0]):
        try:
            standardized_like = _collapse_centering_equivalent_structure(
                structure=structure,
                translations=translations,
                expected_atomic_numbers=fixture.state.bridge.vanilla_atomic_numbers,
            )
        except Exception:
            standardized_like = structure
    try:
        vanilla = map_standardized_structure_to_vanilla_frame(
            standardized_structure=standardized_like,
            vanilla_reference_structure=fixture.state.bridge.vanilla_structure,
            symprec=fixture.state.bridge.symprec,
            angle_tolerance=fixture.state.bridge.angle_tolerance,
        )
    except Exception:
        vanilla = standardized_like
    return vanilla_structure_to_model_tensors(
        structure=vanilla,
        lattice_transform=runner.lattice_transform,
        device=frac_coords.device,
        dtype=frac_coords.dtype,
    )


def _validate_projection_for_case(case: GraphCase, projection, fixture: FixedTemplateFixture | None = None) -> Any:
    structure = _build_vanilla_structure(
        frac_coords=projection.z_frac,
        atomic_numbers=projection.raw.atomic_numbers_chart,
        cell_matrix=projection.raw.cell,
    )
    expected_atomic_numbers = case.atomic_numbers
    if fixture is not None and ORACLE_W_MODE and fixture.uses_oracle_chart:
        expected_atomic_numbers = fixture.chart_atomic_numbers
    return validate_requested_space_group(
        structure=structure,
        requested_space_group=case.requested_sg,
        expected_atomic_numbers=expected_atomic_numbers,
    )


def build_fixed_template_fixture(case: GraphCase) -> FixedTemplateFixture:
    target_cell = cell_from_l(case.gt_lattice, int(case.gt_frac.shape[0])).to(device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    oracle_structure = _build_vanilla_structure(
        frac_coords=case.gt_frac,
        atomic_numbers=case.atomic_numbers,
        cell_matrix=target_cell,
    )

    def _select_candidates(standardization: str):
        return select_requested_template_states(
            frac_coords=case.gt_frac,
            atomic_numbers=case.atomic_numbers,
            cell_matrix=target_cell,
            space_group_number=case.requested_sg,
            standardization=standardization,
            symprec=1e-2,
            angle_tolerance=5.0,
            max_templates=max(FIXTURE_MAX_TEMPLATES, 16),
            template_eval_limit=max(FIXTURE_TEMPLATE_EVAL_LIMIT, 6),
            optimization_steps=FIXTURE_OPT_STEPS,
            learning_rate=FIXTURE_LR,
            coord_weight=1.0,
            lattice_weight=0.0,
            pairdist_weight=0.0,
            steric_weight=0.0,
            volume_weight=0.0,
            k6_weight=0.0,
            freeze_lattice_free_vars=True,
            quick_templates=False,
            top_k=max(TEMPLATE_TOP_K, 6),
            template_prior=template_prior,
            template_prior_weight=0.0,
            oracle_reference_structure=oracle_structure,
            oracle_fit_structure=oracle_structure,
        )

    candidate_entries = []
    selection_errors = []
    for standardization in ['conventional', 'primitive']:
        try:
            states = _select_candidates(standardization)
        except Exception as exc:
            selection_errors.append((standardization, exc))
            continue
        for state0 in states:
            chart_frac, chart_atomic_numbers, chart_cell, chart_k, target_repr, anchor_repr = _pcs_state_chart_target(state0)
            candidate_entries.append({
                'standardization': standardization,
                'state': state0,
                'chart_frac': chart_frac,
                'chart_atomic_numbers': chart_atomic_numbers,
                'chart_cell': chart_cell,
                'chart_k': chart_k,
                'chart_num_atoms': int(chart_atomic_numbers.shape[0]),
                'target_repr': target_repr,
                'anchor_repr': anchor_repr,
                'same_count': int(chart_atomic_numbers.shape[0]) == int(case.gt_frac.shape[0]),
                'expanded': ('expanded' in target_repr) or ('expanded' in anchor_repr) or int(chart_atomic_numbers.shape[0]) != int(case.gt_frac.shape[0]),
            })

    if not candidate_entries:
        lines = [f"{std}:{type(exc).__name__}:{exc}" for std, exc in selection_errors]
        raise RuntimeError(
            f'No fixed-template candidates could be selected for graph={case.graph_id}. selection_errors={lines}'
        )

    same_count_entries = [item for item in candidate_entries if item['same_count']]
    if not same_count_entries:
        candidate_summary = [
            {
                'std': item['standardization'],
                'chart_atoms': item['chart_num_atoms'],
                'target_repr': item['target_repr'],
                'anchor_repr': item['anchor_repr'],
                'template_labels': template_labels(item['state'].template),
            }
            for item in candidate_entries[:8]
        ]
        raise RuntimeError(
            f'No same-count fixed-template candidates for graph={case.graph_id}. candidate_summary={candidate_summary}'
        )

    candidate_entries = same_count_entries
    candidate_entries.sort(
        key=lambda item: (
            0 if item['standardization'] == 'conventional' else 1,
            0 if item['same_count'] else 1,
            float(item['state'].objective),
        )
    )

    candidate_cfgs = [
        FIXED_CFG,
        dataclasses.replace(
            FIXED_CFG,
            projection_config=dataclasses.replace(FIXED_CFG.projection_config, use_fixed_assignment=False),
        ),
    ]
    best_payload = None
    errors = []
    for item in candidate_entries:
        tau0 = torch.zeros(1, 3, device=item['chart_frac'].device, dtype=item['chart_frac'].dtype)
        for cfg0 in candidate_cfgs:
            try:
                projection = project_to_fixed_template(
                    f_frac=item['chart_frac'],
                    atomic_numbers=item['chart_atomic_numbers'],
                    template_state=item['state'],
                    target_k=item['chart_k'],
                    tau0=tau0,
                    config=cfg0,
                )
                J = compute_template_jacobian(projection.theta, projection.raw.state, tau=projection.tau)
                fixture = FixedTemplateFixture(
                    case=case,
                    state=projection.raw.state,
                    theta=projection.theta,
                    tau=projection.tau,
                    z_frac=projection.z_frac,
                    z_k=projection.z_k,
                    z_l=projection.z_l,
                    assignment=projection.assignment,
                    J=J,
                    template_labels=template_labels(projection.raw.state.template),
                    template_multiplicities=template_multiplicities(projection.raw.state.template),
                    template_dofs=template_dofs(projection.raw.state.template),
                    template_total_dof=int(projection.raw.state.template.total_free_dims),
                    requested_sg_match=False,
                    composition_match=False,
                    projection_objective=float(projection.objective),
                    projection_rank=int(getattr(projection.raw, 'ssvd_rank', 0)) if getattr(projection.raw, 'ssvd_rank', None) is not None else None,
                    projection_condition=float(getattr(projection.raw, 'ssvd_condition_number', float('nan'))) if getattr(projection.raw, 'ssvd_condition_number', None) is not None else None,
                    config=cfg0,
                    chart_frac=item['chart_frac'].detach().clone(),
                    chart_atomic_numbers=item['chart_atomic_numbers'].detach().clone(),
                    chart_l=ensure_l_feature(item['chart_cell'], item['chart_num_atoms']),
                    chart_k=item['chart_k'].detach().clone(),
                    chart_num_atoms=item['chart_num_atoms'],
                    raw_num_atoms=int(case.gt_frac.shape[0]),
                    target_representation_name=item['target_repr'],
                    anchor_representation_name=item['anchor_repr'],
                    uses_oracle_chart=bool(ORACLE_W_MODE),
                    oracle_w_payload=_oracle_w_payload_from_state(
                        case,
                        projection.raw.state,
                        chart_num_atoms=item['chart_num_atoms'],
                        chart_repr=item['target_repr'],
                        anchor_repr=item['anchor_repr'],
                    ),
                )
                validation = _validate_projection_for_case(case, projection, fixture)
                coord_residual = float(torch.linalg.norm(wrap_residual(item['chart_frac'], projection.z_frac).reshape(-1)).detach().item())
                score = (
                    0 if validation.requested_space_group_match else 1,
                    0 if validation.composition_match else 1,
                    0 if item['standardization'] == 'conventional' else 1,
                    0 if bool(getattr(cfg0.projection_config, 'use_fixed_assignment', False)) else 1,
                    coord_residual,
                    float(projection.objective),
                )
                if best_payload is None or score < best_payload[0]:
                    best_payload = (score, fixture, validation)
            except Exception as exc:
                errors.append(
                    f"std={item['standardization']} chart_atoms={item['chart_num_atoms']} target_repr={item['target_repr']} "
                    f"fixed={bool(getattr(cfg0.projection_config, 'use_fixed_assignment', False))} {type(exc).__name__}:{exc}"
                )

    if best_payload is None:
        # Fallback: if we have a same-count extracted template state, build W directly from the extractor output
        # and let Gate 1 be the first real projection validity check.
        fallback_item = candidate_entries[0] if candidate_entries else None
        if fallback_item is not None:
            state0 = fallback_item['state']
            tau0 = torch.zeros(1, 3, device=fallback_item['chart_frac'].device, dtype=fallback_item['chart_frac'].dtype)
            theta0 = state0.free_vars.detach().clone()
            J0 = compute_template_jacobian(theta0, state0, tau=tau0)
            assignment0 = state0.fixed_target_assignment.detach().clone() if state0.fixed_target_assignment is not None else torch.arange(int(fallback_item['chart_frac'].shape[0]), device=fallback_item['chart_frac'].device, dtype=torch.long)
            fixture = FixedTemplateFixture(
                case=case,
                state=state0,
                theta=theta0,
                tau=tau0,
                z_frac=fallback_item['chart_frac'].detach().clone(),
                z_k=fallback_item['chart_k'].detach().clone(),
                z_l=ensure_l_feature(fallback_item['chart_cell'], fallback_item['chart_num_atoms']),
                assignment=assignment0,
                J=J0,
                template_labels=template_labels(state0.template),
                template_multiplicities=template_multiplicities(state0.template),
                template_dofs=template_dofs(state0.template),
                template_total_dof=int(state0.template.total_free_dims),
                requested_sg_match=False,
                composition_match=False,
                projection_objective=float(state0.objective),
                projection_rank=None,
                projection_condition=None,
                config=FIXED_CFG,
                chart_frac=fallback_item['chart_frac'].detach().clone(),
                chart_atomic_numbers=fallback_item['chart_atomic_numbers'].detach().clone(),
                chart_l=ensure_l_feature(fallback_item['chart_cell'], fallback_item['chart_num_atoms']),
                chart_k=fallback_item['chart_k'].detach().clone(),
                chart_num_atoms=fallback_item['chart_num_atoms'],
                raw_num_atoms=int(case.gt_frac.shape[0]),
                target_representation_name=fallback_item['target_repr'],
                anchor_representation_name=fallback_item['anchor_repr'],
                uses_oracle_chart=bool(ORACLE_W_MODE),
                oracle_w_payload=_oracle_w_payload_from_state(
                    case,
                    state0,
                    chart_num_atoms=fallback_item['chart_num_atoms'],
                    chart_repr=fallback_item['target_repr'],
                    anchor_repr=fallback_item['anchor_repr'],
                ),
            )
            structure = _build_vanilla_structure(
                frac_coords=fixture.z_frac,
                atomic_numbers=fixture.chart_atomic_numbers,
                cell_matrix=cell_from_l(fixture.z_l, int(fixture.chart_num_atoms)).to(device=fixture.z_frac.device, dtype=fixture.z_frac.dtype),
            )
            validation = validate_requested_space_group(
                structure=structure,
                requested_space_group=case.requested_sg,
                expected_atomic_numbers=fixture.chart_atomic_numbers,
            )
            fixture.requested_sg_match = bool(validation.requested_space_group_match)
            fixture.composition_match = bool(validation.composition_match)
            return fixture
        candidate_summary = [
            {
                'std': item['standardization'],
                'chart_atoms': item['chart_num_atoms'],
                'target_repr': item['target_repr'],
                'anchor_repr': item['anchor_repr'],
                'template_labels': template_labels(item['state'].template),
            }
            for item in candidate_entries[:8]
        ]
        raise RuntimeError(
            f'Could not build an oracle-W fixed template for graph={case.graph_id}. '
            f'candidate_summary={candidate_summary} errors={errors[:8]}'
        )

    _, fixture, validation = best_payload
    fixture.requested_sg_match = bool(validation.requested_space_group_match)
    fixture.composition_match = bool(validation.composition_match)
    return fixture


def _state_to_fixture_chart(case: GraphCase, fixture: FixedTemplateFixture, state: KLDMState) -> KLDMState:
    if not ORACLE_W_MODE or not fixture.uses_oracle_chart:
        return state
    if int(state.f.shape[0]) == int(fixture.chart_num_atoms) and int(state.h.shape[0]) == int(fixture.chart_num_atoms):
        return state
    cell = cell_from_l(state.l, int(state.f.shape[0])).to(device=state.f.device, dtype=state.f.dtype)
    refreshed_state = refresh_pcs_state_anchor(
        state=fixture.state,
        frac_coords=torch.remainder(state.f, 1.0),
        atomic_numbers=state.h.to(device=state.f.device, dtype=torch.long),
        cell_matrix=cell,
        pairdist_weight=0.0,
        pairdist_bins=32,
        pairdist_max_distance=6.0,
        pairdist_bandwidth=0.15,
        coord_weight=1.0,
        lattice_weight=0.0,
        optimization_steps=max(8, min(int(FIXTURE_OPT_STEPS), 32)),
        learning_rate=float(FIXTURE_LR),
    )
    chart_frac, chart_atomic_numbers, chart_cell, chart_k, target_repr, anchor_repr = _pcs_state_chart_target(refreshed_state)
    if int(chart_atomic_numbers.shape[0]) != int(fixture.chart_num_atoms):
        raise RuntimeError(
            f'oracle chart lift mismatch graph={case.graph_id} expected_chart_atoms={fixture.chart_num_atoms} got={int(chart_atomic_numbers.shape[0])} '
            f'target_repr={target_repr!r} anchor_repr={anchor_repr!r} template_atoms={int(refreshed_state.template.total_atoms)}'
        )
    chart_l = ensure_l_feature(chart_cell, int(chart_frac.shape[0]))
    if int(chart_frac.shape[0]) == int(state.v.shape[0]):
        chart_v = state.v.detach().clone()
    else:
        multiplier = max(1, int(chart_frac.shape[0]) // max(int(state.v.shape[0]), 1))
        chart_v = state.v.repeat(multiplier, 1)
    return KLDMState(
        f=chart_frac.detach().clone(),
        v=chart_v.detach().clone(),
        l=chart_l.detach().clone(),
        k=chart_k.detach().clone(),
        h=chart_atomic_numbers.detach().clone().to(device=state.f.device, dtype=torch.long),
        t=state.t,
        dt=state.dt,
        graph_idx0=state.graph_idx0,
        full_state=state.full_state,
        full_times=state.full_times,
    )

def _maybe_lift_state_to_fixture_chart(case: GraphCase, state: KLDMState) -> KLDMState:
    fixture = globals().get('fixtures', {}).get(case.graph_id)
    if fixture is None:
        return state
    return _state_to_fixture_chart(case, fixture, state)


def gt_state_for_graph(case: GraphCase) -> KLDMState:
    f = case.gt_frac.detach().clone()
    l = case.gt_lattice.detach().clone().reshape(-1)
    h = case.atomic_numbers.detach().clone().to(torch.long)
    k = l_to_k(l, case)
    v = torch.zeros_like(f)
    return _maybe_lift_state_to_fixture_chart(case, KLDMState(f=f, v=v, l=l, k=k, h=h, t=0.0, dt=0.0, graph_idx0=case.graph_idx0))


def random_fractional_state(case: GraphCase, *, seed_offset: int = 0) -> KLDMState:
    generator = torch.Generator(device=case.gt_frac.device).manual_seed(SAMPLE_SEED + int(case.graph_id) + int(seed_offset))
    f = torch.rand(case.gt_frac.shape, generator=generator, device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    l = case.gt_lattice.detach().clone().reshape(-1)
    h = case.atomic_numbers.detach().clone().to(torch.long)
    k = l_to_k(l, case)
    v = torch.zeros_like(f)
    return _maybe_lift_state_to_fixture_chart(case, KLDMState(f=f, v=v, l=l, k=k, h=h, t=float('nan'), dt=1.0 / max(1, CAPTURE_N_STEPS), graph_idx0=case.graph_idx0))


def sample_kldm_state_for_graph_at_step(case: GraphCase, *, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS, seed=SAMPLE_SEED) -> KLDMState:
    full_state, times, _step_idx = capture_batch_kldm_state(seed=seed, n_steps=n_steps, capture_step=capture_step)
    return _maybe_lift_state_to_fixture_chart(case, graph_state_from_full(clone_full_state(full_state), case, times=times))


def evaluate_arrays(case: GraphCase, pred_f: torch.Tensor, pred_l: torch.Tensor, pred_h: torch.Tensor) -> dict[str, Any]:
    fixture = globals().get('fixtures', {}).get(case.graph_id)
    eval_f = pred_f
    eval_l = pred_l
    eval_h = pred_h.to(torch.long)
    if fixture is not None and ORACLE_W_MODE and fixture.uses_oracle_chart and int(eval_h.shape[0]) != int(case.atomic_numbers.shape[0]):
        eval_f, eval_l, eval_h = _chart_structure_to_vanilla_tensors(
            case,
            fixture,
            frac_coords=pred_f,
            atomic_numbers=eval_h,
            l=pred_l,
        )
    pred_l_feature = ensure_l_feature(eval_l, int(eval_f.shape[0]))
    result = evaluate_csp_reconstruction(
        pred_f=eval_f,
        pred_l=pred_l_feature,
        pred_a=eval_h,
        target_f=case.gt_frac,
        target_l=case.gt_lattice,
        target_a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
        requested_space_group=case.requested_sg,
        validity_cutoff=float(getattr(BASE_ALGO10, 'hard_min_distance', 0.5)),
    )
    standardized_frac_rmse = None
    if result.matcher_diagnostics is not None:
        standardized_frac_rmse = result.matcher_diagnostics.standardized_frac_rmse
    return {
        'match': bool(result.match),
        'valid': bool(result.valid),
        'rmse': result.rmse,
        'frac_rmse': result.frac_rmse,
        'standardized_frac_rmse': standardized_frac_rmse,
        'composition_match': result.composition_match,
        'requested_sg_match': result.requested_space_group_match,
        'min_pair_distance': result.min_pair_distance,
        'volume': result.volume,
        'lattice_lengths_mae': result.lattice_lengths_mae,
        'lattice_angles_mae': result.lattice_angles_mae,
        'validity_reason': result.validity_reason,
    }


def score_norms_from_modified_state(case: GraphCase, base_state: KLDMState, *, f: torch.Tensor, v: torch.Tensor, l: torch.Tensor, h: torch.Tensor | None = None) -> dict[str, Any]:
    if base_state.full_state is None or base_state.full_times is None:
        return {
            'pred_v_norm': float('nan'),
            'score_v_norm': float('nan'),
            'pred_l_norm': float('nan'),
            'finite_ok': False,
        }
    if int(f.shape[0]) != int(case.gt_frac.shape[0]):
        return {
            'pred_v_norm': float('nan'),
            'score_v_norm': float('nan'),
            'pred_l_norm': float('nan'),
            'finite_ok': False,
        }
    full_state = clone_full_state(base_state.full_state)
    times = base_state.full_times
    start, end = graph_slice(case.graph_idx0)
    full_state['f_t'][start:end] = f.to(device=full_state['f_t'].device, dtype=full_state['f_t'].dtype)
    full_state['v_t'][start:end] = v.to(device=full_state['v_t'].device, dtype=full_state['v_t'].dtype)
    if h is not None and int(torch.as_tensor(h).reshape(-1).numel()) == int(end - start):
        full_state['a_t'][start:end] = torch.as_tensor(h, device=full_state['a_t'].device, dtype=full_state['a_t'].dtype).reshape(end-start)
    l_feature = ensure_l_feature(l, int(case.gt_frac.shape[0]))
    full_state['l_t'][case.graph_idx0] = l_feature.to(device=full_state['l_t'].device, dtype=full_state['l_t'].dtype)
    with torch.no_grad():
        preds = full_state['score_network'](
            t=times.now.graph,
            pos=full_state['f_t'],
            v=full_state['v_t'],
            h=full_state['a_t'],
            l=full_state['l_t'],
            node_index=full_state['node_index'],
            edge_node_index=full_state['edge_node_index'],
        )
        score_v = full_state['sampling_tdm'].reconstruct_full_reverse_velocity_score(
            t=times.now.nodes,
            v_t=full_state['v_t'],
            pred_v=preds['v'],
            index=full_state['node_index'],
        )
    pred_v_graph = preds['v'][start:end]
    score_v_graph = score_v[start:end]
    pred_l_graph = preds['l'][case.graph_idx0].reshape(-1)
    finite_ok = bool(torch.isfinite(pred_v_graph).all() and torch.isfinite(score_v_graph).all() and torch.isfinite(pred_l_graph).all())
    return {
        'pred_v_norm': float(torch.linalg.norm(pred_v_graph.reshape(-1)).detach().item()),
        'score_v_norm': float(torch.linalg.norm(score_v_graph.reshape(-1)).detach().item()),
        'pred_l_norm': float(torch.linalg.norm(pred_l_graph).detach().item()),
        'finite_ok': finite_ok,
    }


def project_state_to_fixture(
    case: GraphCase,
    fixture: FixedTemplateFixture,
    f_frac: torch.Tensor,
    k: torch.Tensor | None = None,
    *,
    use_self_target: bool = False,
    atomic_numbers: torch.Tensor | None = None,
    l_feature: torch.Tensor | None = None,
):
    if atomic_numbers is None:
        if ORACLE_W_MODE and fixture.uses_oracle_chart and int(f_frac.shape[0]) == int(fixture.chart_num_atoms):
            target_atomic_numbers = fixture.chart_atomic_numbers.detach().clone().to(device=f_frac.device, dtype=torch.long)
        else:
            target_atomic_numbers = case.atomic_numbers.detach().clone().to(device=f_frac.device, dtype=torch.long)
    else:
        target_atomic_numbers = atomic_numbers.detach().clone().to(device=f_frac.device, dtype=torch.long)
    if l_feature is None:
        if ORACLE_W_MODE and fixture.uses_oracle_chart and int(f_frac.shape[0]) == int(fixture.chart_num_atoms):
            base_l = fixture.chart_l.detach().clone()
        elif k is not None:
            base_l = k_to_l_num_atoms(k, int(f_frac.shape[0]))
        else:
            base_l = case.gt_lattice
    else:
        base_l = l_feature
    state_like = KLDMState(
        f=f_frac.detach().clone(),
        v=torch.zeros_like(f_frac),
        l=base_l.detach().clone(),
        k=(fixture.chart_k if k is None else k).detach().clone(),
        h=target_atomic_numbers.detach().clone(),
        t=float('nan'),
        dt=0.0,
        graph_idx0=case.graph_idx0,
    )
    if int(state_like.f.shape[0]) != int(fixture.chart_num_atoms) or int(state_like.h.shape[0]) != int(fixture.chart_num_atoms):
        state_like = _state_to_fixture_chart(case, fixture, state_like)
    target_k = (fixture.chart_k if k is None else state_like.k).detach().clone().to(device=state_like.f.device, dtype=state_like.f.dtype)
    template_state = fixture.state
    if use_self_target:
        target_frac = torch.remainder(state_like.f.detach().clone(), 1.0)
        target_l = state_like.l.detach().clone()
        target_cell = cell_from_l(target_l, int(target_frac.shape[0])).to(device=state_like.f.device, dtype=state_like.f.dtype)
        identity = torch.arange(int(target_frac.shape[0]), device=state_like.f.device, dtype=torch.long)
        template_state = dataclasses.replace(
            template_state,
            target_frac=target_frac,
            target_atomic_numbers=state_like.h.detach().clone(),
            target_cell=target_cell.detach().clone(),
            target_k=target_k.detach().clone(),
            fixed_target_assignment=identity.detach().clone(),
            anchor_frac=target_frac.detach().clone(),
            anchor_atomic_numbers=state_like.h.detach().clone(),
            anchor_cell=target_cell.detach().clone(),
            anchor_k=target_k.detach().clone(),
            anchor_assignment=identity.detach().clone(),
        )
    assignment0 = template_state.fixed_target_assignment if template_state.fixed_target_assignment is not None else fixture.assignment
    theta0 = template_state.free_vars if getattr(template_state, 'free_vars', None) is not None else fixture.theta
    projection = project_to_fixed_template_local(
        f_frac=state_like.f,
        atomic_numbers=state_like.h,
        template_state=template_state,
        target_k=target_k,
        tau0=tau_current,
        theta0=theta0,
        fixed_assignment=assignment0,
        config=fixture.config,
    )
    J = compute_template_jacobian(projection.theta, projection.raw.state, tau=projection.tau)
    return projection, J


def template_distance(
    case: GraphCase,
    fixture: FixedTemplateFixture,
    f_frac: torch.Tensor,
    k: torch.Tensor | None = None,
    *,
    use_self_target: bool = False,
    atomic_numbers: torch.Tensor | None = None,
    l_feature: torch.Tensor | None = None,
):
    projection, J = project_state_to_fixture(
        case,
        fixture,
        f_frac,
        k=k,
        use_self_target=use_self_target,
        atomic_numbers=atomic_numbers,
        l_feature=l_feature,
    )
    target_frac = projection.raw.state.target_frac if projection.raw.state.target_frac is not None else fixture.chart_frac
    dist = float(torch.linalg.norm(wrap_residual(target_frac, projection.z_frac).reshape(-1)).detach().item())
    return dist, projection, J


def one_step_repair(case: GraphCase, fixture: FixedTemplateFixture, state: KLDMState, *, mode: str, metric_name: str = 'fractional', eta: float = 0.05):
    state = _state_to_fixture_chart(case, fixture, state)
    projection, J = project_state_to_fixture(case, fixture, state.f, k=state.k, atomic_numbers=state.h, l_feature=state.l)
    metric = metric_tensor(metric_name, projection.z_l.to(device=state.f.device, dtype=state.f.dtype), int(state.f.shape[0]))
    residual = wrap_residual(state.f, projection.z_frac)
    tangent_velocity, projector, mean_norm_before = tangent_project_velocity(
        state.v,
        J=J,
        metric=metric,
        damping=FIXED_CFG.projector_damping,
        mean_free=FIXED_CFG.mean_free_velocity,
    )
    dist_after_projection, _projection_anchor, _J_anchor = template_distance(
        case,
        fixture,
        projection.z_frac,
        k=projection.z_k,
        use_self_target=True,
        atomic_numbers=projection.raw.atomic_numbers_chart,
        l_feature=projection.z_l,
    )
    if mode == 'none':
        f_new = state.f.detach().clone()
        v_new = state.v.detach().clone()
    elif mode == 'final_projection_only':
        f_new = projection.z_frac.detach().clone()
        v_new = state.v.detach().clone()
    elif mode == 'velocity_tangent_only':
        v_new = tangent_velocity
        f_new = kinetic_position_update(state.f, v_new, state.dt)
    elif mode == 'coord_projection_plus_tangent':
        v_new = tangent_velocity
        f_new = kinetic_position_update(projection.z_frac, v_new, state.dt)
    elif mode == 'normal_force_plus_tangent':
        forced = apply_full_space_force(state.v, residual=-residual, step_size=eta, mean_free=False)
        v_new, _projector2, _ = tangent_project_velocity(forced, J=J, metric=metric, damping=FIXED_CFG.projector_damping, mean_free=True)
        f_new = kinetic_position_update(state.f, v_new, state.dt)
    elif mode == 'reduced_space':
        omega = reduced_chart_velocity(state.v, J=J, metric=metric, damping=FIXED_CFG.projector_damping)
        residual_theta = reduced_chart_velocity(residual, J=J, metric=metric, damping=FIXED_CFG.projector_damping)
        omega = apply_reduced_space_force(omega, residual=-residual_theta, step_size=eta)
        v_new = lift_reduced_chart_velocity(omega, J=J, shape=state.v.shape)
        v_new, mean_norm_before = center_velocity(v_new)
        f_new = kinetic_position_update(state.f, v_new, state.dt)
    else:
        raise ValueError(f'unsupported repair mode: {mode}')
    repaired = KLDMState(
        f=f_new,
        v=v_new,
        l=state.l.detach().clone(),
        k=state.k.detach().clone(),
        h=state.h.detach().clone(),
        t=state.t,
        dt=state.dt,
        graph_idx0=state.graph_idx0,
        full_state=state.full_state,
        full_times=state.full_times,
    )
    dist_final, final_projection, final_J = template_distance(case, fixture, repaired.f, k=repaired.k, atomic_numbers=repaired.h, l_feature=repaired.l)
    info = {
        'dist_before': float(torch.linalg.norm(residual.reshape(-1)).detach().item()),
        'dist_after_projection': dist_after_projection,
        'dist_after': dist_final,
        'velocity_norm_before': graph_velocity_norm(state.v),
        'velocity_norm_after': graph_velocity_norm(v_new),
        'mean_norm_before': mean_norm_before,
        'mean_norm_after': mean_norm(v_new),
        'tangent_residual': projector.residual_norm(v_new),
        'projection_objective': float(projection.objective),
        'projection_energy_final': float(final_projection.objective),
        'rank_J': int(projector.rank),
        'cond_J': float(projector.condition_number),
        'min_pair_distance': min_pair_distance_from_arrays(repaired.f, repaired.l, int(repaired.f.shape[0])),
        'volume_ratio': volume_from_l(repaired.l, int(repaired.f.shape[0])) / max(volume_from_l(fixture.chart_l, int(fixture.chart_num_atoms)), 1.0e-8),
    }
    return repaired, projection, J, final_projection, final_J, info


def __run_short_chain_base(case: GraphCase, fixture: FixedTemplateFixture, *, mode: str, start_step: int, end_step: int, metric_name: str = 'fractional', eta: float = 0.05, final_projection: bool = False):
    set_seed(SAMPLE_SEED)
    full_state = runner.model._prepare_csp_sampling(
        batch=batch,
        n_steps=CAPTURE_N_STEPS,
        t_start=float(facit_cfg.get('t_start', 1.0)),
        t_final=float(facit_cfg.get('t_final', 1e-3)),
    )
    dists = []
    projection_fail_count = 0
    start_time = float('nan')
    end_time = float('nan')
    for step_idx, times in enumerate(iter_sampling_times(batch=full_state['batch'], grid=full_state['sampling_time_grid']), start=1):
        if step_idx < start_step:
            full_state = _native_reverse_step_full_state(full_state, times)
            continue
        if step_idx > end_step:
            break
        if math.isnan(start_time):
            start_time = float(times.now.graph[case.graph_idx0].detach().reshape(-1)[0].item())
        full_state = _native_reverse_step_full_state(full_state, times)
        state = graph_state_from_full(full_state, case, times)
        if mode != 'none':
            try:
                repaired, _proj, _J, _proj_final, _J_final, _info = one_step_repair(case, fixture, state, mode=mode, metric_name=metric_name, eta=eta)
                start, end = graph_slice(case.graph_idx0)
                if int(repaired.f.shape[0]) == int(end - start):
                    full_state['f_t'][start:end] = repaired.f.to(device=full_state['f_t'].device, dtype=full_state['f_t'].dtype)
                    full_state['v_t'][start:end] = repaired.v.to(device=full_state['v_t'].device, dtype=full_state['v_t'].dtype)
            except Exception:
                projection_fail_count += 1
        state = graph_state_from_full(full_state, case, times)
        dist_now, _proj_now, _J_now = template_distance(case, fixture, state.f, k=state.k, atomic_numbers=state.h, l_feature=state.l)
        dists.append(dist_now)
        end_time = float(times.now.graph[case.graph_idx0].detach().reshape(-1)[0].item())
    final_state = graph_state_from_full(full_state, case, times)
    if final_projection:
        proj_end, _J_end = project_state_to_fixture(case, fixture, final_state.f, k=final_state.k, atomic_numbers=final_state.h, l_feature=final_state.l)
        final_state = KLDMState(
            f=proj_end.z_frac.detach().clone(),
            v=final_state.v.detach().clone(),
            l=final_state.l.detach().clone(),
            k=proj_end.z_k.detach().clone(),
            h=proj_end.raw.atomic_numbers_chart.detach().clone(),
            t=final_state.t,
            dt=final_state.dt,
            graph_idx0=final_state.graph_idx0,
            full_state=final_state.full_state,
            full_times=final_state.full_times,
        )
        end_dist, _proj_end2, _J_end2 = template_distance(case, fixture, final_state.f, k=final_state.k, atomic_numbers=final_state.h, l_feature=final_state.l)
        dists.append(end_dist)
    evaluation = evaluate_arrays(case, final_state.f, final_state.l, final_state.h)
    score_norms = score_norms_from_modified_state(case, final_state, f=final_state.f, v=final_state.v, l=final_state.l)
    return {
        'dist_initial': dists[0] if dists else float('nan'),
        'dist_final': dists[-1] if dists else float('nan'),
        'projection_fail_count': int(projection_fail_count),
        'start_time': start_time,
        'end_time': end_time,
        'velocity_norm_max': float(max([0.0] + [graph_velocity_norm(final_state.v)])),
        'score_norm_max': float(max(score_norms['score_v_norm'], score_norms['pred_l_norm'])) if score_norms['finite_ok'] else float('nan'),
        **evaluation,
    }


def run_short_chain(case: GraphCase, fixture: FixedTemplateFixture, *, mode: str, start_step: int, end_step: int, metric_name: str = 'fractional', eta: float = 0.05, final_projection: bool = False):
    if ORACLE_W_MODE and fixture.uses_oracle_chart and fixture.chart_num_atoms != fixture.raw_num_atoms:
        dists = []
        projection_fail_count = 0
        start_time = float('nan')
        end_time = float('nan')
        final_state = None
        for step_idx in range(int(start_step), int(end_step) + 1):
            raw_state = sample_kldm_state_for_graph_at_step(case, capture_step=step_idx, n_steps=CAPTURE_N_STEPS)
            state = _state_to_fixture_chart(case, fixture, raw_state)
            if math.isnan(start_time):
                start_time = float(state.t)
            if mode != 'none':
                try:
                    repaired, _proj, _J, _proj_final, _J_final, _info = one_step_repair(case, fixture, state, mode=mode, metric_name=metric_name, eta=eta)
                    state = repaired
                except Exception:
                    projection_fail_count += 1
            dist_now, _proj_now, _J_now = template_distance(case, fixture, state.f, k=state.k, atomic_numbers=state.h, l_feature=state.l)
            dists.append(dist_now)
            end_time = float(state.t)
            final_state = state
        if final_state is None:
            final_state = gt_state_for_graph(case)
        if final_projection:
            proj_end, _J_end = project_state_to_fixture(case, fixture, final_state.f, k=final_state.k, atomic_numbers=final_state.h, l_feature=final_state.l)
            final_state = KLDMState(
                f=proj_end.z_frac.detach().clone(),
                v=final_state.v.detach().clone(),
                l=final_state.l.detach().clone(),
                k=proj_end.z_k.detach().clone(),
                h=proj_end.raw.atomic_numbers_chart.detach().clone(),
                t=final_state.t,
                dt=final_state.dt,
                graph_idx0=final_state.graph_idx0,
                full_state=final_state.full_state,
                full_times=final_state.full_times,
            )
            end_dist, _proj_end2, _J_end2 = template_distance(case, fixture, final_state.f, k=final_state.k, atomic_numbers=final_state.h, l_feature=final_state.l)
            dists.append(end_dist)
        evaluation = evaluate_arrays(case, final_state.f, final_state.l, final_state.h)
        score_norms = score_norms_from_modified_state(case, final_state, f=final_state.f, v=final_state.v, l=final_state.l)
        return {
            'dist_initial': dists[0] if dists else float('nan'),
            'dist_final': dists[-1] if dists else float('nan'),
            'projection_fail_count': int(projection_fail_count),
            'start_time': start_time,
            'end_time': end_time,
            'velocity_norm_max': float(graph_velocity_norm(final_state.v)),
            'score_norm_max': float(max(score_norms['score_v_norm'], score_norms['pred_l_norm'])) if score_norms['finite_ok'] else float('nan'),
            'oracle_shadow_chain': True,
            **evaluation,
        }
    return __run_short_chain_base(
        case,
        fixture,
        mode=mode,
        start_step=start_step,
        end_step=end_step,
        metric_name=metric_name,
        eta=eta,
        final_projection=final_projection,
    )


In [5]:
# Phase-space CASAL helpers

def gt_state_for_graph(case: GraphCase) -> KLDMState:
    f = case.gt_frac.detach().clone()
    v = torch.zeros_like(f)
    l = case.gt_lattice.detach().clone().reshape(-1)
    h = case.atomic_numbers.detach().clone().to(torch.long)
    k = l_to_k(l, case)
    dt = 1.0 / max(CAPTURE_N_STEPS, 1)
    return KLDMState(f=f, v=v, l=l, k=k, h=h, t=float('nan'), dt=dt, graph_idx0=case.graph_idx0)


def phase_space_config(*, rho_f: float, rho_v: float, kappa: float, dual_mode: str) -> PhaseSpaceCasalConfig:
    return PhaseSpaceCasalConfig(
        sign_chi=-1.0,
        projector_damping=1.0e-6,
        mean_free_velocity=True,
        kappa=float(kappa),
        rho_f=float(rho_f),
        rho_v=float(rho_v),
        dual_mode=str(dual_mode),
        projection=FIXED_CFG,
        idempotence_tol=1.0e-3,
        score_ratio_max=3.0,
        velocity_ratio_max=5.0,
        min_pair_distance_min=0.5,
        volume_ratio_min=0.5,
        volume_ratio_max=1.5,
    )


def phase_space_metric_from_state(state: KLDMState) -> torch.Tensor | None:
    return metric_tensor('fractional', state.l.to(device=state.f.device, dtype=state.f.dtype), int(state.f.shape[0]))


def project_phase_space_state(case: GraphCase, fixture: FixedTemplateFixture, state: KLDMState, *, tau0: torch.Tensor | None = None, template_state=None):
    tau_base = fixture.tau if tau0 is None else tau0
    state0 = fixture.state if template_state is None else template_state
    return project_state_to_phase_space_constraint(
        f=state.f,
        v=state.v,
        atomic_numbers=state.h,
        template_state=state0,
        target_k=state.k,
        tau0=tau_base,
        metric=phase_space_metric_from_state(state),
        config=phase_space_config(rho_f=1.0, rho_v=0.0, kappa=0.1, dual_mode='no_dual'),
    )


def score_norms_from_arrays(case: GraphCase, base_state: KLDMState, *, f: torch.Tensor, v: torch.Tensor, l: torch.Tensor, h: torch.Tensor | None = None):
    return score_norms_from_modified_state(case, base_state, f=f, v=v, l=l, h=(base_state.h if h is None else h))


def kldm_proposal_pair(case: GraphCase, *, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS, seed=SAMPLE_SEED):
    pre_step = max(int(capture_step) - 1, 1)
    full_state, times, _ = capture_batch_kldm_state(seed=seed, n_steps=n_steps, capture_step=pre_step)
    pre_raw = graph_state_from_full(clone_full_state(full_state), case, times)
    step_state = clone_full_state(full_state)
    step_state = _native_reverse_step_full_state(step_state, times)
    post_raw = graph_state_from_full(step_state, case, times)
    pre_state = _maybe_lift_state_to_fixture_chart(case, pre_raw)
    post_state = _maybe_lift_state_to_fixture_chart(case, post_raw)
    return pre_state, post_state


def phase_space_step_from_states(case: GraphCase, fixture: FixedTemplateFixture, pre_state: KLDMState, proposal_state: KLDMState, *, rho_f: float, rho_v: float, kappa: float, dual_mode: str, template_state=None, tau_current: torch.Tensor | None = None):
    score_before = score_norms_from_arrays(case, pre_state, f=pre_state.f, v=pre_state.v, l=pre_state.l, h=pre_state.h)
    score_after = score_norms_from_arrays(case, proposal_state, f=proposal_state.f, v=proposal_state.v, l=proposal_state.l, h=proposal_state.h)
    out = phase_space_casal_step(
        f=pre_state.f,
        v=pre_state.v,
        proposal_velocity=proposal_state.v,
        atomic_numbers=pre_state.h,
        template_state=(fixture.state if template_state is None else template_state),
        target_k=pre_state.k,
        tau_current=(fixture.tau if tau_current is None else tau_current),
        h=max(abs(float(pre_state.dt)), 1.0e-8),
        metric=phase_space_metric_from_state(pre_state),
        mu_f=torch.zeros_like(pre_state.f),
        mu_v=torch.zeros_like(pre_state.v),
        score_norm_before=float(score_before['score_v_norm']),
        score_norm_after=float(score_after['score_v_norm']),
        config=phase_space_config(rho_f=rho_f, rho_v=rho_v, kappa=kappa, dual_mode=dual_mode),
    )
    return out, score_before, score_after


def run_phase_space_synthetic_chain(case: GraphCase, fixture: FixedTemplateFixture, *, rho_f: float, rho_v: float, kappa: float, dual_mode: str, num_steps: int = 20):
    state = sample_kldm_state_for_graph_at_step(case, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS, seed=SAMPLE_SEED)
    mu_f = torch.zeros_like(state.f)
    mu_v = torch.zeros_like(state.v)
    template_state_current = fixture.state
    tau_current = fixture.tau
    d_f = []
    d_v = []
    mu_f_norm = []
    mu_v_norm = []
    vel_norm = []
    idem = []
    for _ in range(int(num_steps)):
        proj = project_state_to_phase_space_constraint(
            f=state.f,
            v=state.v,
            atomic_numbers=state.h,
            template_state=template_state_current,
            target_k=state.k,
            tau0=tau_current,
            metric=phase_space_metric_from_state(state),
            config=phase_space_config(rho_f=rho_f, rho_v=rho_v, kappa=kappa, dual_mode=dual_mode),
        )
        out = phase_space_casal_step(
            f=state.f,
            v=state.v,
            proposal_velocity=state.v,
            atomic_numbers=state.h,
            template_state=template_state_current,
            target_k=state.k,
            tau_current=tau_current if proj is None else proj.tau,
            h=max(abs(float(state.dt)), 1.0e-8),
            metric=phase_space_metric_from_state(state),
            mu_f=mu_f,
            mu_v=mu_v,
            config=phase_space_config(rho_f=rho_f, rho_v=rho_v, kappa=kappa, dual_mode=dual_mode),
        )
        state = KLDMState(f=out.z_f_next.detach().clone(), v=out.z_v_next.detach().clone(), l=state.l.detach().clone(), k=state.k.detach().clone(), h=state.h.detach().clone(), t=state.t, dt=state.dt, graph_idx0=case.graph_idx0)
        mu_f, mu_v = out.mu_f_next.detach().clone(), out.mu_v_next.detach().clone()
        template_state_current = out.projection_new.projection.raw.state
        tau_current = out.projection_new.tau.detach().clone()
        d_f.append(out.projection_new.d_f)
        d_v.append(out.projection_new.d_v)
        mu_f_norm.append(float(torch.linalg.norm(mu_f.reshape(-1)).detach().item()))
        mu_v_norm.append(float(torch.linalg.norm(mu_v.reshape(-1)).detach().item()))
        vel_norm.append(float(torch.linalg.norm(state.v.reshape(-1)).detach().item()))
        idem.append(float(out.projection_new.idempotence_error))
    return {
        'num_steps': int(num_steps),
        'D_f_initial': float(d_f[0]) if d_f else float('nan'),
        'D_f_final': float(d_f[-1]) if d_f else float('nan'),
        'D_v_initial': float(d_v[0]) if d_v else float('nan'),
        'D_v_final': float(d_v[-1]) if d_v else float('nan'),
        'mu_f_norm_max': float(max(mu_f_norm)) if mu_f_norm else float('nan'),
        'mu_v_norm_max': float(max(mu_v_norm)) if mu_v_norm else float('nan'),
        'velocity_norm_max': float(max(vel_norm)) if vel_norm else float('nan'),
        'idempotence_error_max': float(max(idem)) if idem else float('nan'),
    }


def run_phase_space_kldm_chain(case: GraphCase, fixture: FixedTemplateFixture, *, start_step: int, end_step: int, rho_f: float, rho_v: float, kappa: float, dual_mode: str, apply_every_step: bool = True, final_projection_only: bool = False):
    set_seed(SAMPLE_SEED)
    full_state = runner.model._prepare_csp_sampling(
        batch=batch,
        n_steps=int(max(end_step, 1)),
        t_start=float(facit_cfg.get('t_start', 1.0)),
        t_final=float(facit_cfg.get('t_final', 1e-3)),
    )
    shadow_state = None
    mu_f = None
    mu_v = None
    template_state_current = fixture.state
    tau_current = fixture.tau
    d_f_hist = []
    d_v_hist = []
    score_hist = []
    accepted_count = 0
    projection_fail_count = 0
    for step_idx, times in enumerate(iter_sampling_times(batch=full_state['batch'], grid=full_state['sampling_time_grid']), start=1):
        raw_state = graph_state_from_full(clone_full_state(full_state), case, times)
        if shadow_state is None:
            shadow_state = raw_state
        full_state = _native_reverse_step_full_state(full_state, times)
        proposal_state = graph_state_from_full(clone_full_state(full_state), case, times)
        if step_idx < int(start_step):
            shadow_state = proposal_state
            continue
        try:
            if final_projection_only and step_idx >= int(end_step):
                proj = project_state_to_phase_space_constraint(
                    f=proposal_state.f,
                    v=proposal_state.v,
                    atomic_numbers=proposal_state.h,
                    template_state=template_state_current,
                    target_k=proposal_state.k,
                    tau0=tau_current,
                    metric=phase_space_metric_from_state(proposal_state),
                    config=phase_space_config(rho_f=rho_f, rho_v=rho_v, kappa=kappa, dual_mode='no_dual'),
                )
                shadow_state = KLDMState(f=proj.z_f.detach().clone(), v=proj.z_v.detach().clone(), l=proposal_state.l.detach().clone(), k=proposal_state.k.detach().clone(), h=proposal_state.h.detach().clone(), t=proposal_state.t, dt=proposal_state.dt, graph_idx0=case.graph_idx0)
                template_state_current = proj.projection.raw.state
                tau_current = proj.tau.detach().clone()
                d_f_hist.append(proj.d_f)
                d_v_hist.append(proj.d_v)
            elif apply_every_step:
                out = phase_space_casal_step(
                    f=shadow_state.f,
                    v=shadow_state.v,
                    proposal_velocity=proposal_state.v,
                    atomic_numbers=shadow_state.h,
                    template_state=template_state_current,
                    target_k=shadow_state.k,
                    tau_current=tau_current,
                    h=max(abs(float(shadow_state.dt)), 1.0e-8),
                    metric=phase_space_metric_from_state(shadow_state),
                    mu_f=(torch.zeros_like(shadow_state.f) if mu_f is None else mu_f),
                    mu_v=(torch.zeros_like(shadow_state.v) if mu_v is None else mu_v),
                    config=phase_space_config(rho_f=rho_f, rho_v=rho_v, kappa=kappa, dual_mode=dual_mode),
                )
                if out.accepted:
                    accepted_count += 1
                    shadow_state = KLDMState(f=out.z_f_next.detach().clone(), v=out.z_v_next.detach().clone(), l=proposal_state.l.detach().clone(), k=proposal_state.k.detach().clone(), h=proposal_state.h.detach().clone(), t=proposal_state.t, dt=proposal_state.dt, graph_idx0=case.graph_id - 1)
                    mu_f, mu_v = out.mu_f_next.detach().clone(), out.mu_v_next.detach().clone()
                    template_state_current = out.projection_new.projection.raw.state
                    tau_current = out.projection_new.tau.detach().clone()
                    d_f_hist.append(out.projection_new.d_f)
                    d_v_hist.append(out.projection_new.d_v)
                else:
                    shadow_state = proposal_state
                    projection_fail_count += 1
            else:
                shadow_state = proposal_state
            score_hist.append(float(score_norms_from_arrays(case, proposal_state, f=shadow_state.f, v=shadow_state.v, l=shadow_state.l, h=shadow_state.h)['score_v_norm']))
        except Exception:
            projection_fail_count += 1
            shadow_state = proposal_state
        if step_idx >= int(end_step):
            break
    final_eval = evaluate_arrays(case, shadow_state.f, shadow_state.l, shadow_state.h) if shadow_state is not None else {}
    final_proj = project_state_to_phase_space_constraint(
        f=shadow_state.f,
        v=shadow_state.v,
        atomic_numbers=shadow_state.h,
        template_state=template_state_current,
        target_k=shadow_state.k,
        tau0=tau_current,
        metric=phase_space_metric_from_state(shadow_state),
        config=phase_space_config(rho_f=rho_f, rho_v=rho_v, kappa=kappa, dual_mode=dual_mode),
    ) if shadow_state is not None else None
    return {
        'n_steps': max(0, int(end_step) - int(start_step) + 1),
        'D_f_final': float(final_proj.d_f) if final_proj is not None else float('nan'),
        'D_v_final': float(final_proj.d_v) if final_proj is not None else float('nan'),
        'score_norm_max': float(max(score_hist)) if score_hist else float('nan'),
        'velocity_norm_max': float(torch.linalg.norm(shadow_state.v.reshape(-1)).detach().item()) if shadow_state is not None else float('nan'),
        'projection_fail_count': int(projection_fail_count),
        'accepted_count': int(accepted_count),
        'frac_rmse': final_eval.get('frac_rmse', float('nan')),
        'rmse': final_eval.get('rmse', float('nan')),
        'match': final_eval.get('match', False),
        'requested_sg_match': final_eval.get('requested_sg_match', False),
        'valid': final_eval.get('valid', False),
        'min_pair_distance': min_pair_distance_from_arrays(shadow_state.f, shadow_state.l, int(shadow_state.f.shape[0])) if shadow_state is not None else float('nan'),
        'volume_ratio': volume_from_l(shadow_state.l, int(shadow_state.f.shape[0])) / max(volume_from_l(fixture.z_l, int(fixture.z_frac.shape[0])), 1.0e-12) if shadow_state is not None else float('nan'),
    }


In [6]:
# Gate 0: fixed-template setup
fixtures = {}
rows = []
for case in GRAPH_CASES:
    try:
        fixture = build_fixed_template_fixture(case)
        fixtures[case.graph_id] = fixture
        template_valid = bool(int(fixture.chart_num_atoms) == int(case.gt_frac.shape[0]))
        rows.append({
            'test': '0_fixed_template_setup',
            'graph_id': case.graph_id,
            'space_group': case.requested_sg,
            'num_atoms_graph': int(case.gt_frac.shape[0]),
            'num_atoms_template': int(fixture.chart_num_atoms),
            'template_labels': fixture.template_labels,
            'theta_dim': fixture.template_total_dof,
            'template_valid': template_valid,
            'status': 'PASS' if template_valid else 'REJECT',
            'PASS': template_valid,
        })
    except Exception as exc:
        msg = str(exc)
        if 'No same-count fixed-template candidates' in msg:
            rows.append({
                'test': '0_fixed_template_setup',
                'graph_id': case.graph_id,
                'space_group': case.requested_sg,
                'num_atoms_graph': int(case.gt_frac.shape[0]),
                'num_atoms_template': float('nan'),
                'template_labels': '',
                'theta_dim': float('nan'),
                'template_valid': False,
                'status': 'REJECT',
                'PASS': False,
                'failure_category': 'TEMPLATE_COUNT_MISMATCH',
                'error_message': msg,
            })
        else:
            rows.append(error_row('0_fixed_template_setup', case.graph_id, exc, 'FIXTURE_BUILD_ERROR', space_group=case.requested_sg))
df_0 = dataframe_result('0_fixed_template_setup', rows)
display(df_0)
DYNAMICS_CASES = [case for case in GRAPH_CASES if case.graph_id in fixtures and int(fixtures[case.graph_id].chart_num_atoms) == int(case.gt_frac.shape[0])]
print('dynamics_graphs=', [case.graph_id for case in DYNAMICS_CASES])


,test,graph_id,space_group,num_atoms_graph,num_atoms_template,template_labels,theta_dim,template_valid,status,PASS,failure_category,error_message
0,0_fixed_template_setup,1,227,6,NaN,,NaN,False,REJECT,False,TEMPLATE_COUNT_MISMATCH,No same-count fixed-template candidates for gr...
1,0_fixed_template_setup,2,4,16,16.0,"26@2a,26@2a,26@2a,27@2a,27@2a,27@2a,5@2a,5@2a",24.0,True,PASS,True,NaN,NaN
2,0_fixed_template_setup,3,194,8,8.0,"72@2a,22@6h",1.0,True,PASS,True,NaN,NaN


dynamics_graphs= [2, 3]


In [7]:
# Gates 1-3: coordinate projection, Jacobian/tangent sanity, initial residual diagnostics
rows_1 = []
rows_2 = []
rows_3 = []
for case in DYNAMICS_CASES:
    fixture = fixtures[case.graph_id]
    try:
        for source_name, state in [('gt', gt_state_for_graph(case)), ('kldm', sample_kldm_state_for_graph_at_step(case, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS, seed=SAMPLE_SEED))]:
            proj = project_phase_space_state(case, fixture, state)
            validation = _validate_projection_for_case(case, proj.projection, fixture)
            rows_1.append({
                'test': '1_coordinate_projection_sanity',
                'graph': case.graph_id,
                'source': source_name,
                'D_f': proj.d_f,
                'idempotence_error': proj.idempotence_error,
                'projection_energy': float(proj.projection.objective),
                'projection_success': bool(proj.projection_success),
                'requested_sg_match': bool(validation.requested_space_group_match),
                'min_pair_distance': float(proj.projection.min_pair_distance),
                'PASS': bool(proj.projection_success and proj.idempotence_error < 1.0e-3 and torch.isfinite(proj.z_f).all().item()),
                'status': 'PASS' if bool(proj.projection_success and proj.idempotence_error < 1.0e-3 and torch.isfinite(proj.z_f).all().item()) else 'FAIL',
            })
        state = sample_kldm_state_for_graph_at_step(case, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS, seed=SAMPLE_SEED)
        proj = project_phase_space_state(case, fixture, state)
        if fixture.template_total_dof > 0:
            direction = sample_random_free_vars(fixture.state.template, device=proj.theta.device, dtype=proj.theta.dtype).reshape(-1)
            if direction.numel() != proj.theta.numel():
                direction = torch.randn_like(proj.theta)
        else:
            direction = torch.zeros_like(proj.theta)
        if proj.theta.numel() == 0:
            fd_abs = 0.0
            fd_rel = 0.0
        else:
            fd_abs, fd_rel = finite_difference_jacobian_error(theta=proj.theta, direction=direction, epsilon=1.0e-4, template_state=proj.projection.raw.state, tau=proj.tau, jacobian=proj.jacobian)
        centered_v, mean_velocity_norm = center_velocity_impulse(state.v)
        rank_ratio = float(proj.rank_J / max(3 * int(state.f.shape[0]), 1))
        rows_2.append({
            'test': '2_jacobian_and_tangent_projection_sanity',
            'graph': case.graph_id,
            'fd_jacobian_error': float(fd_rel),
            'rank_J': int(proj.rank_J),
            'cond_J': float(proj.cond_J),
            'velocity_norm': float(torch.linalg.norm(centered_v.reshape(-1)).detach().item()),
            'zv_norm': float(torch.linalg.norm(proj.z_v.reshape(-1)).detach().item()),
            'D_v': float(proj.d_v),
            'velocity_survival_ratio': float(proj.velocity_survival_ratio),
            'mean_velocity_norm': float(mean_velocity_norm),
            'rank_ratio': rank_ratio,
            'PASS': bool(np.isfinite(fd_rel) and fd_rel < 1.0e-2 and torch.isfinite(proj.z_v).all().item()),
            'status': 'PASS' if bool(np.isfinite(fd_rel) and fd_rel < 1.0e-2 and torch.isfinite(proj.z_v).all().item()) else 'FAIL',
            'failure_label': 'LOW_RANK_VELOCITY_COLLAPSE' if float(proj.velocity_survival_ratio) < 0.05 else '',
        })
        rows_3.append({
            'test': '3_initial_residual_diagnostic',
            'graph': case.graph_id,
            'D_f_initial': float(proj.d_f),
            'D_v_initial': float(proj.d_v),
            'D_f_over_D_v': float(proj.d_f / max(proj.d_v, 1.0e-12)),
            'rank_ratio': rank_ratio,
            'velocity_survival_ratio': float(proj.velocity_survival_ratio),
            'PASS': bool(np.isfinite(proj.d_f) and np.isfinite(proj.d_v)),
            'status': 'PASS' if bool(np.isfinite(proj.d_f) and np.isfinite(proj.d_v)) else 'FAIL',
        })
    except Exception as exc:
        rows_1.append(error_row('1_coordinate_projection_sanity', case.graph_id, exc, 'COORD_PROJECTION_FAIL'))
        rows_2.append(error_row('2_jacobian_and_tangent_projection_sanity', case.graph_id, exc, 'JACOBIAN_FAIL'))
        rows_3.append(error_row('3_initial_residual_diagnostic', case.graph_id, exc, 'INITIAL_RESIDUAL_FAIL'))
df_1 = dataframe_result('1_coordinate_projection_sanity', rows_1)
df_2 = dataframe_result('2_jacobian_and_tangent_projection_sanity', rows_2)
df_3 = dataframe_result('3_initial_residual_diagnostic', rows_3)
display(df_1)
display(df_2)
display(df_3)


,test,graph,source,D_f,idempotence_error,projection_energy,projection_success,requested_sg_match,min_pair_distance,PASS,status
0,1_coordinate_projection_sanity,2,gt,2.038591,2.019103,1.441782e-11,True,True,2.036511,False,FAIL
1,1_coordinate_projection_sanity,2,kldm,1.812505,2.021579,4.741864e-02,True,True,1.147274,False,FAIL
2,1_coordinate_projection_sanity,3,gt,1.272167,0.430175,2.238027e-02,True,True,2.014342,False,FAIL
3,1_coordinate_projection_sanity,3,kldm,1.124737,0.050727,5.522122e-02,True,True,0.103022,False,FAIL


,test,graph,fd_jacobian_error,rank_J,cond_J,velocity_norm,zv_norm,D_v,velocity_survival_ratio,mean_velocity_norm,rank_ratio,PASS,status,failure_label
0,2_jacobian_and_tangent_projection_sanity,2,0.000284,23,1.0,0.728792,0.561170,0.465001,0.770000,8.981350e-09,0.479167,True,PASS,
1,2_jacobian_and_tangent_projection_sanity,3,0.000247,1,1.0,0.512791,0.015451,0.512559,0.030131,1.130128e-08,0.041667,True,PASS,LOW_RANK_VELOCITY_COLLAPSE


,test,graph,D_f_initial,D_v_initial,D_f_over_D_v,rank_ratio,velocity_survival_ratio,PASS,status
0,3_initial_residual_diagnostic,2,1.812505,0.465001,3.897849,0.479167,0.770000,True,PASS
1,3_initial_residual_diagnostic,3,1.124737,0.512559,2.194358,0.041667,0.030131,True,PASS


In [8]:
# Gates 4-6: one-step phase-space CASAL, KLDM proposal diagnostics, one-step sweep
rows_4 = []
rows_5 = []
rows_6 = []
rows_6b = []
for case in DYNAMICS_CASES:
    fixture = fixtures[case.graph_id]
    try:
        state = sample_kldm_state_for_graph_at_step(case, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS, seed=SAMPLE_SEED)
        proj0 = project_phase_space_state(case, fixture, state)
        out0 = phase_space_casal_step(
            f=state.f,
            v=state.v,
            proposal_velocity=state.v,
            atomic_numbers=state.h,
            template_state=(fixture.state if template_state is None else template_state),
            target_k=state.k,
            tau_current=(fixture.tau if tau_current is None else tau_current),
            h=max(abs(float(state.dt)), 1.0e-8),
            metric=phase_space_metric_from_state(state),
            mu_f=torch.zeros_like(state.f),
            mu_v=torch.zeros_like(state.v),
            config=phase_space_config(rho_f=1.0, rho_v=0.1, kappa=0.1, dual_mode='mu_f_and_mu_v'),
        )
        pass4 = bool((out0.diagnostics.d_f_after_casal < out0.diagnostics.d_f_before + EPS_PASS and out0.diagnostics.d_v_after_casal < out0.diagnostics.d_v_before + EPS_PASS) or ((out0.diagnostics.d_f_after_casal < out0.diagnostics.d_f_before + EPS_PASS or out0.diagnostics.d_v_after_casal < out0.diagnostics.d_v_before + EPS_PASS) and out0.diagnostics.d_f_after_casal <= 1.5 * out0.diagnostics.d_f_before + EPS_PASS and out0.diagnostics.d_v_after_casal <= 1.5 * out0.diagnostics.d_v_before + EPS_PASS))
        rows_4.append({
            'test': '4_no_score_phase_space_casal_only',
            'graph': case.graph_id,
            'D_f_old': out0.diagnostics.d_f_before,
            'D_v_old': out0.diagnostics.d_v_before,
            'D_f_new': out0.diagnostics.d_f_after_casal,
            'D_v_new': out0.diagnostics.d_v_after_casal,
            'PASS': pass4,
            'status': 'PASS' if pass4 else 'FAIL',
        })
        pre_state, proposal_state = kldm_proposal_pair(case, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS, seed=SAMPLE_SEED)
        proposal_proj = project_phase_space_state(case, fixture, proposal_state)
        proposal_score = score_norms_from_arrays(case, proposal_state, f=proposal_state.f, v=proposal_state.v, l=proposal_state.l, h=proposal_state.h)
        rows_5.append({
            'test': '5_kldm_proposal_plus_phase_space_projection',
            'graph': case.graph_id,
            'D_f_kldm': proposal_proj.d_f,
            'D_v_kldm': proposal_proj.d_v,
            'score_norm_kldm': float(proposal_score['score_v_norm']),
            'velocity_norm_hat': float(torch.linalg.norm(proposal_state.v.reshape(-1)).detach().item()),
            'PASS': bool(np.isfinite(proposal_proj.d_f) and np.isfinite(proposal_proj.d_v)),
            'status': 'PASS' if bool(np.isfinite(proposal_proj.d_f) and np.isfinite(proposal_proj.d_v)) else 'FAIL',
        })
        out, score_before, score_after = phase_space_step_from_states(case, fixture, pre_state, proposal_state, rho_f=1.0, rho_v=0.1, kappa=0.1, dual_mode='mu_f_and_mu_v')
        min_pair_distance = min_pair_distance_from_arrays(out.z_f_next, pre_state.l, int(pre_state.f.shape[0]))
        volume_ratio = volume_from_l(pre_state.l, int(pre_state.f.shape[0])) / max(volume_from_l(fixture.z_l, int(fixture.z_frac.shape[0])), 1.0e-12)
        pass6 = bool(out.diagnostics.d_f_after_casal < out.diagnostics.d_f_after_kldm + EPS_PASS and out.diagnostics.d_v_after_casal < out.diagnostics.d_v_after_kldm + EPS_PASS and (not np.isfinite(out.diagnostics.score_ratio) or out.diagnostics.score_ratio < 3.0) and min_pair_distance > 0.5)
        rows_6.append({
            'test': '6_one_full_phase_space_casal_step',
            'graph': case.graph_id,
            'rho_f': 1.0,
            'rho_v': 0.1,
            'tau': out.diagnostics.tau,
            'h': out.diagnostics.h,
            'D_f_before': out.diagnostics.d_f_before,
            'D_v_before': out.diagnostics.d_v_before,
            'D_f_after_kldm': out.diagnostics.d_f_after_kldm,
            'D_v_after_kldm': out.diagnostics.d_v_after_kldm,
            'D_f_after_casal': out.diagnostics.d_f_after_casal,
            'D_v_after_casal': out.diagnostics.d_v_after_casal,
            'velocity_norm_before': out.diagnostics.velocity_norm_before,
            'velocity_norm_after': out.diagnostics.velocity_norm_after,
            'score_norm_before': out.diagnostics.score_norm_before,
            'score_norm_after': out.diagnostics.score_norm_after,
            'score_ratio': out.diagnostics.score_ratio,
            'projection_success': out.diagnostics.projection_success,
            'idempotence_error': out.diagnostics.idempotence_error,
            'min_pair_distance': min_pair_distance,
            'volume_ratio': volume_ratio,
            'accepted': bool(out.accepted),
            'failure_label': out.failure_label,
            'PASS': pass6,
            'status': 'PASS' if pass6 else 'FAIL',
        })
        for rho_f in RHO_F_SWEEP:
            for rho_v in [0.0, 0.01 * rho_f, 0.1 * rho_f, 1.0 * rho_f]:
                for kappa in KAPPA_SWEEP:
                    for dual_mode in DUAL_MODES:
                        out_sweep, _, _ = phase_space_step_from_states(case, fixture, pre_state, proposal_state, rho_f=rho_f, rho_v=rho_v, kappa=kappa, dual_mode=dual_mode)
                        pass6b = bool(out_sweep.diagnostics.d_f_after_casal < out_sweep.diagnostics.d_f_after_kldm + EPS_PASS and out_sweep.diagnostics.d_v_after_casal < out_sweep.diagnostics.d_v_after_kldm + EPS_PASS)
                        rows_6b.append({
                            'test': '6b_one_step_hyperparameter_sweep',
                            'graph': case.graph_id,
                            'rho_f': rho_f,
                            'rho_v': rho_v,
                            'kappa': kappa,
                            'dual_mode': dual_mode,
                            'D_f_reduction': out_sweep.diagnostics.d_f_reduction,
                            'D_v_reduction': out_sweep.diagnostics.d_v_reduction,
                            'score_ratio': out_sweep.diagnostics.score_ratio,
                            'accepted': bool(out_sweep.accepted),
                            'failure_label': out_sweep.failure_label,
                            'PASS': pass6b,
                            'status': 'PASS' if pass6b else 'DIAG',
                        })
    except Exception as exc:
        rows_4.append(error_row('4_no_score_phase_space_casal_only', case.graph_id, exc, 'PHASE_SPACE_CASAL_ONLY_ERROR'))
        rows_5.append(error_row('5_kldm_proposal_plus_phase_space_projection', case.graph_id, exc, 'KLDM_PROPOSAL_ERROR'))
        rows_6.append(error_row('6_one_full_phase_space_casal_step', case.graph_id, exc, 'ONE_STEP_PHASE_SPACE_ERROR'))
        rows_6b.append(error_row('6b_one_step_hyperparameter_sweep', case.graph_id, exc, 'ONE_STEP_SWEEP_ERROR'))
df_4 = dataframe_result('4_no_score_phase_space_casal_only', rows_4)
df_5 = dataframe_result('5_kldm_proposal_plus_phase_space_projection', rows_5)
df_6 = dataframe_result('6_one_full_phase_space_casal_step', rows_6)
df_6b = dataframe_result('6b_one_step_hyperparameter_sweep', rows_6b)
display(df_4)
display(df_5)
display(df_6)
display(df_6b)


,test,graph,PASS,status,error_type,error_message,traceback_tail,failure_category
0,4_no_score_phase_space_casal_only,2,False,ERROR,NameError,name 'template_state' is not defined,NameError: name 'template_state' is not defined,PHASE_SPACE_CASAL_ONLY_ERROR
1,4_no_score_phase_space_casal_only,3,False,ERROR,NameError,name 'template_state' is not defined,NameError: name 'template_state' is not defined,PHASE_SPACE_CASAL_ONLY_ERROR


,test,graph,PASS,status,error_type,error_message,traceback_tail,failure_category
0,5_kldm_proposal_plus_phase_space_projection,2,False,ERROR,NameError,name 'template_state' is not defined,NameError: name 'template_state' is not defined,KLDM_PROPOSAL_ERROR
1,5_kldm_proposal_plus_phase_space_projection,3,False,ERROR,NameError,name 'template_state' is not defined,NameError: name 'template_state' is not defined,KLDM_PROPOSAL_ERROR


,test,graph,PASS,status,error_type,error_message,traceback_tail,failure_category
0,6_one_full_phase_space_casal_step,2,False,ERROR,NameError,name 'template_state' is not defined,NameError: name 'template_state' is not defined,ONE_STEP_PHASE_SPACE_ERROR
1,6_one_full_phase_space_casal_step,3,False,ERROR,NameError,name 'template_state' is not defined,NameError: name 'template_state' is not defined,ONE_STEP_PHASE_SPACE_ERROR


,test,graph,PASS,status,error_type,error_message,traceback_tail,failure_category
0,6b_one_step_hyperparameter_sweep,2,False,ERROR,NameError,name 'template_state' is not defined,NameError: name 'template_state' is not defined,ONE_STEP_SWEEP_ERROR
1,6b_one_step_hyperparameter_sweep,3,False,ERROR,NameError,name 'template_state' is not defined,NameError: name 'template_state' is not defined,ONE_STEP_SWEEP_ERROR


In [ ]:
# Gates 7-9 and final summary
rows_7 = []
rows_8 = []
rows_9 = []
for case in DYNAMICS_CASES:
    fixture = fixtures[case.graph_id]
    try:
        for label, rho_f, rho_v, dual_mode in [
            ('coordinate_only', 1.0, 0.0, 'mu_f_only'),
            ('phase_space_small_rhov', 1.0, 0.1, 'mu_f_and_mu_v'),
            ('phase_space_large_rhov', 1.0, 1.0, 'mu_f_and_mu_v'),
        ]:
            summary = run_phase_space_synthetic_chain(case, fixture, rho_f=rho_f, rho_v=rho_v, kappa=0.1, dual_mode=dual_mode, num_steps=20)
            pass7 = bool(summary['D_f_final'] < summary['D_f_initial'] + EPS_PASS and summary['D_v_final'] < summary['D_v_initial'] + EPS_PASS and summary['idempotence_error_max'] < 1.0e-3)
            rows_7.append({
                'test': '7_synthetic_multi_step',
                'graph': case.graph_id,
                'label': label,
                'rho_f': rho_f,
                'rho_v': rho_v,
                **summary,
                'PASS': pass7,
                'status': 'PASS' if pass7 else 'FAIL',
            })
        for method, rho_f, rho_v, dual_mode, final_projection_only in [
            ('KLDM_baseline', 0.0, 0.0, 'no_dual', False),
            ('KLDM_plus_coordinate_only_CASAL', 1.0, 0.0, 'mu_f_only', False),
            ('KLDM_plus_phase_space_CASAL_small', 1.0, 0.1, 'mu_f_and_mu_v', False),
            ('KLDM_plus_phase_space_CASAL_large', 1.0, 1.0, 'mu_f_and_mu_v', False),
            ('KLDM_plus_final_projection_only', 1.0, 0.1, 'no_dual', True),
        ]:
            summary = run_phase_space_kldm_chain(case, fixture, start_step=max(1, LATE_START_STEP), end_step=min(CAPTURE_N_STEPS, LATE_START_STEP + MINI_CHAIN_STEPS - 1), rho_f=rho_f, rho_v=rho_v, kappa=0.1, dual_mode=dual_mode, apply_every_step=(method != 'KLDM_baseline'), final_projection_only=final_projection_only)
            pass8 = bool(summary['valid'] and summary['requested_sg_match'] and summary['projection_fail_count'] == 0)
            rows_8.append({
                'test': '8_late_mini_chain_with_kldm',
                'graph': case.graph_id,
                'method': method,
                'rho_f': rho_f,
                'rho_v': rho_v,
                **summary,
                'PASS': pass8,
                'status': 'PASS' if pass8 else 'DIAG',
            })
        for method, rho_f, rho_v, dual_mode, final_projection_only in [
            ('KLDM_baseline', 0.0, 0.0, 'no_dual', False),
            ('KLDM_plus_final_projection', 1.0, 0.1, 'no_dual', True),
            ('KLDM_plus_phase_space_CASAL', 1.0, 0.1, 'mu_f_and_mu_v', False),
            ('KLDM_plus_phase_space_CASAL_plus_final_zout', 1.0, 0.1, 'mu_f_and_mu_v', True),
        ]:
            summary = run_phase_space_kldm_chain(case, fixture, start_step=1, end_step=min(CAPTURE_N_STEPS, FULL_CHAIN_STEPS), rho_f=rho_f, rho_v=rho_v, kappa=0.1, dual_mode=dual_mode, apply_every_step=(method != 'KLDM_baseline'), final_projection_only=final_projection_only)
            pass9 = bool(summary['valid'] and summary['requested_sg_match'] and summary['match'])
            rows_9.append({
                'test': '9_full_reverse_chain',
                'graph': case.graph_id,
                'method': method,
                'rho_f': rho_f,
                'rho_v': rho_v,
                **summary,
                'PASS': pass9,
                'status': 'PASS' if pass9 else 'FAIL',
            })
    except Exception as exc:
        rows_7.append(error_row('7_synthetic_multi_step', case.graph_id, exc, 'SYNTHETIC_MULTISTEP_ERROR'))
        rows_8.append(error_row('8_late_mini_chain_with_kldm', case.graph_id, exc, 'MINICHAIN_ERROR'))
        rows_9.append(error_row('9_full_reverse_chain', case.graph_id, exc, 'FULL_CHAIN_ERROR'))

df_7 = dataframe_result('7_synthetic_multi_step', rows_7)
df_8 = dataframe_result('8_late_mini_chain_with_kldm', rows_8)
df_9 = dataframe_result('9_full_reverse_chain', rows_9)
display(df_7)
display(df_8)
display(df_9)

summary_row = {
    'COORD_PROJECTION_VALID': bool((not df_1.empty) and df_1['PASS'].all()),
    'TANGENT_PROJECTION_VALID': bool((not df_2.empty) and df_2['PASS'].all()),
    'INITIAL_DF_DV_MEANINGFUL': bool((not df_3.empty) and df_3['PASS'].all()),
    'ONE_STEP_DF_REDUCED': bool((not df_6.empty) and (df_6['D_f_after_casal'] < df_6['D_f_after_kldm']).all()),
    'ONE_STEP_DV_REDUCED': bool((not df_6.empty) and (df_6['D_v_after_casal'] < df_6['D_v_after_kldm']).all()),
    'SCORE_SAFE': bool((not df_6.empty) and (df_6['score_ratio'].fillna(0.0) < 3.0).all()),
    'SYNTHETIC_MULTISTEP_STABLE': bool((not df_7.empty) and df_7['PASS'].any()),
    'KLDM_MINICHAIN_STABLE': bool((not df_8.empty) and df_8['PASS'].any()),
    'FULL_CHAIN_VALID': bool((not df_9.empty) and df_9['PASS'].any()),
}
if summary_row['FULL_CHAIN_VALID']:
    recommendation = 'phase_space_casal_viable'
elif summary_row['ONE_STEP_DF_REDUCED'] and summary_row['ONE_STEP_DV_REDUCED']:
    recommendation = 'needs_chart_training'
elif summary_row['ONE_STEP_DF_REDUCED']:
    recommendation = 'coordinate_only_casal'
else:
    recommendation = 'final_projection_only'
summary_row['RECOMMENDATION'] = recommendation
summary_row['CASAL_THEOREM_APPLIES'] = CASAL_THEOREM_APPLIES
summary_row['REASON'] = CASAL_THEOREM_REASON

df_summary = dataframe_result('final_summary', [summary_row])
display(df_summary)
